# Step 4 — vLLM & SGLang: High-Throughput Inference

## Why do we need specialised inference servers?

A naive HuggingFace `.generate()` call has several bottlenecks:

| Problem | Cause | Solution |
|---------|-------|----------|
| Low GPU utilisation | Sequential requests, KV cache allocated per-request | Continuous batching |
| Memory waste | KV cache pre-allocated for max_length | PagedAttention (vLLM) |
| Redundant computation | Prefix computed again per request | Prefix caching |
| Complex pipelines | No native support for branching/loops | SGLang program model |

---

## vLLM — PagedAttention + Continuous Batching

**Core innovations**:

### 1. PagedAttention
Traditional attention allocates a contiguous KV cache block per sequence:
```
Request A: [K1 K2 K3 K4 _ _ _ _ ]  ← 4 used, 4 wasted (reserved for max_len)
Request B: [K1 K2 _ _ _ _ _ _ ]    ← 2 used, 6 wasted
```
PagedAttention uses virtual pages (like OS virtual memory):
```
Physical KV blocks: [B1][B2][B3][B4][B5][B6]   ← all used, none wasted
Req A page table:    B1  B2  B3  B4
Req B page table:    B5  B6
→ Near-zero fragmentation, 2-4× more concurrent requests
```

### 2. Continuous Batching
Instead of static batches (process B requests, wait for all to finish):
```
Static:     [===A====][====B=====]   ← GPU waits for longest request
Continuous: [=A=][C][=B=][D][==E==] ← new requests inserted as slots free
→ 23× higher throughput (Kwon et al., 2023)
```

---

## SGLang — Structured Generation Language

SGLang adds a **programming model** on top of efficient inference:
- **RadixAttention**: prefix caching at the token level (shared system prompts cached once)
- **Constrained decoding**: enforce JSON schema / regex on outputs
- **Parallelism primitives**: `fork()`, `join()` for branching generation
- **Runtime compilation**: traces your Python program into an optimised execution plan

```python
# SGLang example — multi-choice scoring
@sgl.function
def score_choices(s, question, choices):
    s += sgl.system("You are a helpful assistant.")
    s += sgl.user(question)
    forks = s.fork(len(choices))          # parallel branches
    for i, f in enumerate(forks):
        f += sgl.assistant(sgl.gen(f"score_{i}", max_tokens=1))
    forks.join()                           # sync
```

**When to use which**:
- **vLLM**: Maximum raw throughput, OpenAI-compatible API, production serving
- **SGLang**: Complex pipelines (RAG, multi-step reasoning, structured output, multi-agent)

In [ ]:
# Installations:
# pip install vllm           (requires CUDA GPU)
# pip install sglang[all]    (requires CUDA GPU)
# pip install openai         (for OpenAI-compatible client)

import os, time, json, subprocess
from pathlib import Path

MODEL_DIR = "../models"
MERGED    = f"{MODEL_DIR}/lora_merged"

# Note on architecture:
# vLLM natively supports: LLaMA, Mistral, Falcon, OPT, GPT-NeoX, T5, BART, and more.
# For our roberta2roberta model, vLLM supports it via the seq2seq backend.
print("vLLM & SGLang inference notebook")
print("=" * 50)
print("For this demo we show two paths:")
print("  A) roberta2roberta via vLLM seq2seq backend")
print("  B) Use a decoder-only model (e.g. opt-1.3b) — better vLLM support")

## vLLM — Path A: Start the server

vLLM exposes an OpenAI-compatible REST API. You start it once, then hit it
from any client (Python, curl, any language).

In [ ]:
# ── Option A: roberta2roberta (seq2seq) via vLLM Python API ──────────────
# vLLM supports encoder-decoder via its seq2seq engine path.

try:
    from vllm import LLM, SamplingParams

    print("Initialising vLLM engine for seq2seq model…")
    # vLLM auto-detects architecture from config.json
    llm = LLM(
        model=MERGED,
        dtype="float16",        # use bfloat16 on Ampere+
        max_model_len=512,
        gpu_memory_utilization=0.85,  # fraction of GPU VRAM for KV cache
        # tensor_parallel_size=2,    # split across 2 GPUs if available
    )

    sampling_params = SamplingParams(
        max_tokens=128,
        temperature=0.0,        # greedy (deterministic)
        # temperature=0.7,      # sampling (creative)
        # top_p=0.9,            # nucleus sampling
    )

    articles = [
        "Scientists have discovered a new species of deep-sea fish near hydrothermal vents.",
        "The stock market fell sharply today as inflation data came in higher than expected.",
    ]

    t0 = time.perf_counter()
    outputs = llm.generate(articles, sampling_params)
    elapsed = (time.perf_counter() - t0) * 1000

    for output in outputs:
        print(f"Input : {output.prompt[:60]}…")
        print(f"Output: {output.outputs[0].text}")
        print()

    print(f"vLLM batch inference: {elapsed:.1f} ms for {len(articles)} samples")
    print(f"Throughput: {len(articles) / (elapsed/1000):.1f} samples/sec")

except ImportError:
    print("vLLM not installed (requires CUDA GPU).")
    print("Run:  pip install vllm")

## vLLM — Path B: OpenAI-compatible REST server

Start the server from terminal (or via subprocess), then use any HTTP client.

In [ ]:
# ── Server startup command (run this in a separate terminal) ──────────────
print("""Start vLLM server:

  python -m vllm.entrypoints.openai.api_server \\
    --model ../models/lora_merged \\
    --host 0.0.0.0 \\
    --port 8000 \\
    --dtype float16 \\
    --max-model-len 512

Then in Python:""")

# ── Client code (OpenAI SDK) ──────────────────────────────────────────────
vllm_client_code = '''
from openai import OpenAI
import time

# vLLM speaks the OpenAI API protocol
client = OpenAI(
    api_key="EMPTY",              # vLLM doesn't need a real key
    base_url="http://localhost:8000/v1",
)

# ── Chat completions (for instruct/chat models) ───────────────────────────
response = client.chat.completions.create(
    model="../models/lora_merged",
    messages=[
        {"role": "system", "content": "Summarize the following article concisely."},
        {"role": "user",   "content": "Scientists discovered a new exoplanet…"},
    ],
    max_tokens=128,
    temperature=0.0,
)
print(response.choices[0].message.content)

# ── Batch requests (concurrent) ───────────────────────────────────────────
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

async def batch_summarise(articles: list[str]) -> list[str]:
    tasks = [
        async_client.chat.completions.create(
            model="../models/lora_merged",
            messages=[{"role": "user", "content": art}],
            max_tokens=128,
        )
        for art in articles
    ]
    results = await asyncio.gather(*tasks)
    return [r.choices[0].message.content for r in results]

articles = ["Article 1…", "Article 2…", "Article 3…"]
summaries = asyncio.run(batch_summarise(articles))
'''
print(vllm_client_code)

## SGLang — Structured Generation

SGLang is particularly powerful for:
1. **Structured JSON outputs** — guaranteed valid JSON
2. **Multi-step pipelines** — think-before-answer, RAG
3. **Branching** — generate multiple options in parallel, pick the best

In [ ]:
# ── Start SGLang server (terminal) ────────────────────────────────────────
print("""Start SGLang server:

  python -m sglang.launch_server \\
    --model-path ../models/lora_merged \\
    --host 0.0.0.0 \\
    --port 30000 \\
    --dtype float16

SGLang client code:""")

sglang_code = '''
import sglang as sgl

# Set the backend runtime
sgl.set_default_backend(sgl.RuntimeEndpoint("http://localhost:30000"))

# ── Example 1: Simple summarisation ──────────────────────────────────────
@sgl.function
def summarise(s, article: str):
    s += sgl.system("You are an expert news summariser. Be concise.")
    s += sgl.user(f"Article: {article}")
    s += sgl.assistant(sgl.gen("summary", max_tokens=128, temperature=0.0))

state = summarise.run(article="Scientists found water ice on the Moon…")
print(state["summary"])


# ── Example 2: Structured JSON output ────────────────────────────────────
@sgl.function
def extract_entities(s, article: str):
    s += sgl.system("Extract key entities as JSON.")
    s += sgl.user(article)
    s += sgl.assistant(
        sgl.gen(
            "entities",
            max_tokens=256,
            # Constrained decoding: output MUST match this JSON schema
            json_schema={
                "type": "object",
                "properties": {
                    "people":        {"type": "array", "items": {"type": "string"}},
                    "organisations": {"type": "array", "items": {"type": "string"}},
                    "locations":     {"type": "array", "items": {"type": "string"}},
                },
            }
        )
    )
import json
entities = json.loads(state["entities"])


# ── Example 3: Chain-of-Thought + Answer ────────────────────────────────
@sgl.function
def cot_summarise(s, article: str):
    s += sgl.user(f"Article: {article}")
    s += sgl.assistant(
        "Key points:" +
        sgl.gen("key_points", max_tokens=100, stop="\n\n") +
        "\n\nSummary: " +
        sgl.gen("summary", max_tokens=64)
    )
# state["key_points"] and state["summary"] are both accessible


# ── Example 4: Parallel batch with fork ──────────────────────────────────
@sgl.function
def multi_angle_summary(s, article: str):
    """Generate 3 summaries from different angles in parallel."""
    s += sgl.user(f"Article: {article}")
    perspectives = ["technical", "business", "general public"]
    forks = s.fork(len(perspectives))
    for fork, perspective in zip(forks, perspectives):
        fork += sgl.assistant(
            f"({perspective} angle) " + sgl.gen(f"summary_{perspective}", max_tokens=80)
        )
    forks.join()   # wait for all branches to complete

# RadixAttention automatically caches the shared article prefix
# so each branch only computes from the divergence point.
'''
print(sglang_code)

## Performance Comparison: Naive vs vLLM vs SGLang

In [ ]:
import numpy as np
import pandas as pd

# Simulated benchmark data based on published results
# (replace with actual measurements from your hardware)
comparison = pd.DataFrame([
    {
        "Framework":       "HuggingFace .generate()",
        "Throughput (req/s)": 8,
        "GPU Util %":      35,
        "Features":        "Simple, no optimisation",
    },
    {
        "Framework":       "vLLM",
        "Throughput (req/s)": 185,
        "GPU Util %":      92,
        "Features":        "PagedAttn, continuous batching, OpenAI API",
    },
    {
        "Framework":       "SGLang",
        "Throughput (req/s)": 210,
        "GPU Util %":      94,
        "Features":        "RadixAttn, prefix cache, structured gen, fork/join",
    },
])

print(comparison.to_string(index=False))
print()
print("Rule of thumb:")
print("  Use vLLM   → pure inference throughput, OpenAI-drop-in replacement")
print("  Use SGLang → complex pipelines, shared prefixes, structured output")